# Pre-processing

## Strut'n'Tie — LUSAS LPI Course Series  
### Video 04

**Author:** Kamil Riedel  
**© Strut'n'Tie**  
**License:** MIT Licence

# Connect to LUSAS

In [2]:
from shared.LPI import *
import shared.Helpers as Helpers

modeller = get_lusas_modeller()
Helpers.initialise(modeller)

if not modeller.existsDatabase():
    raise Exception("A model must be open before running this code")

database = modeller.database()

# Define variables

In [3]:
# ================================================================== #
# STRING CONSTANTS #
BOTTOM_CHORD = "bottom chord"
TOP_CHORD = "top chord"
DIAGONALS = "diagonals"
FIXED = "fixed"
ROLLER = "roller"
ANALYSIS_MAIN = "Main Analysis"
LOADCASE_SW = "Self-weight"
LOADCASE_DL = "Dead load"
LOADCASE_LL = "Live load"

In [4]:
# ================================================================== #
# DICTIONARIES TO DEFINE TRUSS #
points_coord = { 1: (0.0, 0), 2: (1.5, 3), 3: (3.0, 0), 4: (4.5, 3), 5: (6.0, 0), 6: (7.5, 3), 7: (9.0, 0), 8: (10.5, 3), 9: (12.0, 0),     10: (13.5, 3), 11: (15.0, 0),    12: (16.5, 3), 13: (18.0, 0),    14: (19.5, 3), 15: (21.0, 0),    16: (22.5, 3), 17: (24.0, 0),    18: (25.5, 3), 19: (27.0, 0) }
elm_connectivity = { 1:(1,3), 2:(3,5), 3:(5,7), 4:(7,9), 5:(9,11), 6:(11,13), 7:(13,15), 8:(15,17), 9:(17,19), 10:(2,4), 11:(4,6), 12:(6,8), 13:(8,10), 14:(10,12), 15:(12,14), 16:(14,16), 17:(16,18), 18:(1,2), 19:(2,3), 20:(3,4), 21:(4,5), 22:(5,6), 23:(6,7), 24:(7,8), 25:(8,9), 26:(9,10), 27:(10,11), 28:(11,12), 29:(12,13), 30:(13,14), 31:(14,15), 32:(15,16), 33:(16,17), 34:(17,18), 35:(18,19) }
sections = { 1: BOTTOM_CHORD, 2: BOTTOM_CHORD, 3: BOTTOM_CHORD, 4: BOTTOM_CHORD, 5: BOTTOM_CHORD, 6: BOTTOM_CHORD, 7: BOTTOM_CHORD, 8: BOTTOM_CHORD, 9: BOTTOM_CHORD, 10: TOP_CHORD, 11: TOP_CHORD, 12: TOP_CHORD, 13: TOP_CHORD, 14: TOP_CHORD, 15: TOP_CHORD, 16: TOP_CHORD, 17: TOP_CHORD, 18: DIAGONALS, 19: DIAGONALS, 20: DIAGONALS, 21: DIAGONALS, 22: DIAGONALS, 23: DIAGONALS, 24: DIAGONALS, 25: DIAGONALS, 26: DIAGONALS, 27: DIAGONALS, 28: DIAGONALS, 29: DIAGONALS, 30: DIAGONALS, 31: DIAGONALS, 32: DIAGONALS, 33: DIAGONALS, 34: DIAGONALS, 35: DIAGONALS }
supports = {1 : FIXED, 19 : ROLLER}
loads = [3,5,7,9,11,13,15,17]

# Analyses

In [5]:
# ================================================================== #
# ANALYSES #

# Rename the first analysis
analysis = database.getAnalyses()[0]
analysis.setName(ANALYSIS_MAIN)

# Rename the first loadcase
loadCase_SW = database.getLoadsets("All", "All")[0]
loadCase_SW.setName(LOADCASE_SW)
# Enable gravity in self-weight loadcase
loadCase_SW.addGravity(True)
loadCase_SW.setGravityFactor(1.0)

# Create additional loadcases
loadCase_DL = database.createLoadcase(LOADCASE_DL, ANALYSIS_MAIN)
loadCase_LL = database.createLoadcase(LOADCASE_LL, ANALYSIS_MAIN)

# Define Geometry

In [6]:
# ================================================================== #
# DEFINE GEOMETRY #

# Create points
# geometry_data = modeller.geometryData().setAllDefaults()
# geometry_data.setLowerOrderGeometryType("coordinates")
# for (x, y) in points_coord.values():
#     geometry_data.addCoords(x, y, 0.0)
# object_set = database.createPoint(geometry_data)
# points = object_set.getObjects("Point")

points = {}
for pointID, (x, y) in points_coord.items():
    points[pointID] = Helpers.create_point(x, y, 0.0)

# Create lines
lines = {}
for elmID, (p1ID, p2ID) in elm_connectivity.items():
    point1, point2 = points[p1ID], points[p2ID]
    point1, point2 = database.getObject("Point", point1), database.getObject("Point", point2)
    lines[elmID] = Helpers.create_line_from_points(point1, point2)

# Create attributes

In [ ]:
# ================================================================== #
# CREATE ATTRIBUTES #
meshAttr = database.createMeshLine("Beam Mesh")
meshAttr.setSize("BMI21", 1)
meshAttr.setEndRelease("Start", "THY", "free")
meshAttr.setEndRelease("Start", "THZ", "free")
meshAttr.setEndRelease("End", "THY", "free")
meshAttr.setEndRelease("End", "THZ", "free")

geomAttrChords = database.createGeometricLine(BOTTOM_CHORD)
geomAttrChords.setValue("elementType", "3D Thick Beam")
geomAttrChords.setFromLibrary("UK Sections", "Universal Columns (BS4)", "254x254x73kg UC", 0, 0, 0)

geomAttrChords = database.createGeometricLine(TOP_CHORD)
geomAttrChords.setValue("elementType", "3D Thick Beam")
geomAttrChords.setFromLibrary("UK Sections", "Universal Columns (BS4)", "254x254x73kg UC", 0, 0, 0)

geomAttrDiag = database.createGeometricLine(DIAGONALS)
geomAttrDiag.setValue("elementType", "3D Thick Beam")
geomAttrDiag.setFromLibrary("UK Sections", "Universal Columns (BS4)", "152x152x30kg UC", 0, 0, 0)

matAttr = database.createIsotropicMaterial("Iso1", 210.0E9, 0.3, 7.84913E3).setValue("alpha", 12.0E-6, 0)
matAttr.setDefinitionMenuID(1, None, True)
matAttr.setDescription("Ungraded | Steel - Structural | EN1993-1-1:2005")
matAttr = database.getAttribute("Isotropic Material", "Iso1")
matAttr.createValue("Region", 0, 0, 0, 0, 0, 0, 0).setValue("Region", "UK")
matAttr.createValue("Standard", 0, 0, 0, 0, 0, 0, 0).setValue("Standard", "EN1993-1-1:2005")
matAttr.createValue("Material", 0, 0, 0, 0, 0, 0, 0).setValue("Material", "Steel - Structural")
matAttr.createValue("Grade", 0, 0, 0, 0, 0, 0, 0).setValue("Grade", "Ungraded")

supportFixed = database.createSupportStructural(FIXED)
supportFixed.setStructural("R", "R", "R", "R", "R", "R", "F", "F", "C", "F")

supportRoller = database.createSupportStructural(ROLLER)
supportRoller.setStructural("R", "R", "R", "F", "F", "F", "F", "F", "C", "F")

loadAttrDL = database.createLoadingConcentrated("DL")
loadAttrDL.setConcentrated(0.0, "-1.0E7", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)

loadAttrLL = database.createLoadingConcentrated("LL")
loadAttrLL.setConcentrated(0.0, "-1.5E7", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)

<COMObject setConcentrated>

# Assign attributes

In [8]:
# ================================================================== #
# ASSIGN ATTRIBUTES #
for ID, line in lines.items():
    # We need to rotate mesh by 90 degrees
    # meshAttr.assignTo(line, 1)

    sessionFileAssignment = modeller.getSessionFileAssignment()
    sessionFileAssignment.setAllDefaults()
    sessionFileAssignment.setLoadset(loadCase_SW)
    sessionFileAssignment.setBetaAngle("90.0")
    sessionFileAssignment.meshSecondaryToPrimary()
    sessionFileAssignment.setSingleFeatureJointOrient("axes")
    meshAttr.assignTo(line, sessionFileAssignment)
    # Update the mesh to apply the changes
    database.updateMesh()

    matAttr.assignTo(line, 1)

    # if sections[ID]==BOTTOM_CHORD:
    #     geomAttrChords.assignTo(line, 1)
    # elif sections[ID]==TOP_CHORD:
    #     geomAttrChords.assignTo(line, 1)
    # else:
    #     geomAttrDiag.assignTo(line, 1)
    geomAttr = database.getAttribute("Geometric", sections[ID])
    geomAttr.assignTo(line, 1)

# Assign supports
for pointID, supportName in supports.items():
    supportAttr = database.getAttribute("Support", supportName)
    supportAttr.assignTo(points[pointID], 1)
    
# Assign point loads
assignDL = modeller.newAssignment().setAllDefaults()
assignDL.setLoadset(loadCase_DL)  
assignLL = modeller.newAssignment().setAllDefaults()
assignLL.setLoadset(loadCase_LL) 
for pointID in loads:
    loadAttrDL.assignTo(points[pointID], assignDL)
    loadAttrLL.assignTo(points[pointID], assignLL)



# Solve

In [10]:
# ================================================================== #
# SOLVE #

# Run the analysis
database.getAnalysis(ANALYSIS_MAIN).solve(False)

# Open available results
database.openAllResults(False)